# Assessment 2
## Question 9

The following code creates a Bayesian Network for the Earthquake example we looked at in lesson 3.

Create an appropriate query to estimate the probability of a burglary (given that John and Mary both called) using rejection sampling


In [1]:
#This code is modified from the examples at pgmpy.org Copyright (c) 2013-2021 pgmpy
#Importing Library
from pgmpy.models import BayesianModel
from pgmpy.factors.discrete import State
from pgmpy.sampling import BayesianModelSampling
from pgmpy.sampling import GibbsSampling


C:\Users\wesle\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#Defining network structure

alarm_model = BayesianModel([('Burglary', 'Alarm'),
                              ('Earthquake', 'Alarm'),
                              ('Alarm', 'JohnCalls'),
                              ('Alarm', 'MaryCalls')])

C:\Users\wesle\AppData\Local\Programs\Python\Python311\Lib\site-packages\pgmpy\models\BayesianModel.py:8: FutureWarning: BayesianModel has been renamed to BayesianNetwork. Please use BayesianNetwork class, BayesianModel will be removed in future.
  warnings.warn(


In [3]:
#Defining the parameters using CPT
from pgmpy.factors.discrete import TabularCPD

cpd_burglary = TabularCPD(variable='Burglary', variable_card=2,
                      values=[[.999], [0.001]])
cpd_earthquake = TabularCPD(variable='Earthquake', variable_card=2,
                       values=[[0.998], [0.002]])
cpd_alarm = TabularCPD(variable='Alarm', variable_card=2,
                        values=[[0.999, 0.71, 0.06, 0.05],
                                [0.001, 0.29, 0.94, 0.95]],
                        evidence=['Burglary', 'Earthquake'],
                        evidence_card=[2, 2])
cpd_johncalls = TabularCPD(variable='JohnCalls', variable_card=2,
                      values=[[0.95, 0.1], [0.05, 0.9]],
                      evidence=['Alarm'], evidence_card=[2])
cpd_marycalls = TabularCPD(variable='MaryCalls', variable_card=2,
                      values=[[0.99, 0.3], [0.01, 0.7]],
                      evidence=['Alarm'], evidence_card=[2])

# Associating the parameters with the model structure
alarm_model.add_cpds(cpd_burglary, cpd_earthquake, cpd_alarm, cpd_johncalls, cpd_marycalls)

C:\Users\wesle\AppData\Local\Temp\ipykernel_9080\3295705617.py:8: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  cpd_alarm = TabularCPD(variable='Alarm', variable_card=2,
C:\Users\wesle\AppData\Local\Temp\ipykernel_9080\3295705617.py:13: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  cpd_johncalls = TabularCPD(variable='JohnCalls', variable_card=2,
C:\Users\wesle\AppData\Local\Temp\ipykernel_9080\3295705617.py:16: DeprecationWarning: `product` is deprecated as of NumPy 1.25.0, and will be removed in NumPy 2.0. Please use `prod` instead.
  cpd_marycalls = TabularCPD(variable='MaryCalls', variable_card=2,


In [4]:
from pgmpy.sampling import BayesianModelSampling 

In [5]:
model = BayesianModelSampling(alarm_model)

In [10]:
evidence = [State(var='JohnCalls', state=1), State(var='MaryCalls', state=1)]
df = model.rejection_sample(evidence=evidence, size=1000)



  0%|                                                                                         | 0/1000 [00:00<?, ?it/s]C:\Users\wesle\AppData\Local\Programs\Python\Python311\Lib\site-packages\pgmpy\sampling\base.py:609: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and/or drop to modify the DataFrame instead.
  df = pandas.DataFrame.from_records(samples)
C:\Users\wesle\AppData\Local\Programs\Python\Python311\Lib\site-packages\pgmpy\sampling\base.py:609: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and/or drop to modify the DataFrame instead.
  df = pandas.DataFrame.from_records(samples)


 32%|█████████████████████████                                                     | 321/1000 [00:00<00:01, 666.44it/s]C:\Users\wesle\AppData\Local\Programs\Python\Python311\Lib\site-packages\pgmpy\sampling\base.py:609: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and

In [11]:
df.head()

,Burglary,Alarm,Earthquake,JohnCalls,MaryCalls
0,1,1,0,1,1
1,0,1,1,1,1
2,1,1,0,1,1
3,0,1,0,1,1
4,0,0,0,1,1


In [16]:
df['Burglary'].sum() / df.shape[0]

0.26